# CenterNet (ResNet34+FPN) — Train cắt chữ Hán–Nôm trên Kaggle GPU

**Trước khi chạy:**
1. Settings → Accelerator = **GPU T4 x2** (⚠️ KHÔNG dùng **P100** — torch mới của Kaggle không hỗ trợ sm_60), Internet = **ON**.
2. Add data → attach dataset bạn upload từ `train_crop/kaggle_pkg.zip` (đã gồm **code + Nôm + MTH** trong 1 gói).
3. Add-ons → Secrets → thêm `HF_TOKEN` (token Write) để tự đẩy ckpt lên HuggingFace.
4. **Save Version → Save & Run All (Commit)** để pretrain MTH (Cell A) + fine-tune Nôm (Cell B) chạy liền mạch (~4–5h, đóng máy vẫn xong).

> Kaggle có sẵn torch + torchvision + opencv. AMP tự bật trên CUDA. **Pretrain MTH (Cell A) là BẮT BUỘC**; Cell B warm-start từ nó.

In [ ]:
# 0) GPU + tìm gói code đã attach (hỗ trợ cả khi Kaggle giữ nguyên .zip), copy về /kaggle/working
import glob, os, shutil, zipfile, subprocess, warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # tắt cảnh báo + thanh upload HF -> đỡ lag UI
print(subprocess.getoutput('nvidia-smi -L'))
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

def find_pkg():
    hits = glob.glob('/kaggle/input/**/train_centernet.py', recursive=True)
    return os.path.dirname(hits[0]) if hits else None

PKG = find_pkg()
if PKG is None:  # Kaggle có thể giữ nguyên .zip -> tự giải nén
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        print('giải nén', z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall('/kaggle/working/_pkg')
    hits = glob.glob('/kaggle/working/**/train_centernet.py', recursive=True)
    PKG = os.path.dirname(hits[0]) if hits else None
assert PKG, 'Không thấy train_centernet.py — đã attach dataset chưa?'
print('PKG =', PKG)
for f in glob.glob(PKG + '/*'):
    dst = '/kaggle/working/' + os.path.basename(f)
    shutil.copytree(f, dst, dirs_exist_ok=True) if os.path.isdir(f) else shutil.copy(f, dst)
os.chdir('/kaggle/working')
print('files:', sorted(os.listdir('.'))[:12])

In [ ]:
# 1) Smoke test nhanh (build + forward + backward + decode) — nên thấy 'smoke OK'
!python train_centernet.py --smoke
!python infer_centernet.py --smoke

## Bước A — Pretrain MTH/TKH (BẮT BUỘC, ~1.08M box)
Cùng miền ván khắc CJK → bắt buộc để CenterNet học tách chữ dính trước. Dữ liệu đã đóng gói
sẵn (`mth_manifest.json`). Cell A pretrain 35 epoch → `mthv2_pretrain.best.pt`; Cell B bắt buộc
warm-start từ ckpt này. (Tổng A+B ~4–5h trên T4 → dùng **Save Version → Commit**.)

In [ ]:
# A) Pretrain MTH/TKH (BẮT BUỘC) — chạy luôn. ckpt -> mthv2_pretrain.best.pt
import os, json
assert os.path.exists('/kaggle/working/mth_manifest.json'), \
    'Thiếu mth_manifest.json! Dataset phải có MTH (upload lại kaggle_pkg.zip mới).'
m = json.load(open('mth_manifest.json'))
print('Pretrain MTH:', len(m), 'trang |', sum(x['n_boxes'] for x in m), 'box')
!python train_centernet.py --manifest mth_manifest.json \
    --img 768 --epochs 35 --batch 8 --workers 2 --val-frac 0.02 --val-pages 30 --log-every 0 \
    --out /kaggle/working/mthv2_pretrain.pt --hf-repo mdnt571/nom-char-det-pretrain

## Bước B — Fine-tune trên Nôm (BẮT BUỘC warm-start từ MTH)
Bắt buộc `--init mthv2_pretrain.best.pt` (Cell A phải chạy xong). Train `--img 1024 --lr 5e-5 --dcn`,
tự đẩy ckpt lên HF. Dùng **Save Version → Commit** để A+B chạy liền mạch tới 12h.

In [ ]:
# B) Fine-tune Nôm (BẮT BUỘC warm-start từ pretrain MTH).
import os
PRE = '/kaggle/working/mthv2_pretrain.best.pt'
assert os.path.exists(PRE), 'Chưa có pretrain — chạy Cell A (pretrain MTH) trước đã!'
!python train_centernet.py \
    --manifest detect_manifest.json \
    --img 1024 --epochs 40 --batch 4 --workers 2 --lr 5e-5 --dcn \
    --val-frac 0.1 --val-pages 44 --log-every 0 \
    --out /kaggle/working/detector_r34.pt \
    --hf-repo mdnt571/nom-char-det --init $PRE
# OOM thì giảm --batch 2.

In [ ]:
# 2) Kết quả: ckpt + chỉ số VAL đã lưu trong checkpoint
import torch, glob
for p in sorted(glob.glob('/kaggle/working/*r34*.pt') + glob.glob('/kaggle/working/mthv2_pretrain*.pt')):
    d = torch.load(p, map_location='cpu')
    print(p.split('/')[-1], '| epoch', d.get('epoch'), '| VAL', d.get('val'))
print('\nTải về: Output panel bên phải -> detector_r34.best.pt (hoặc từ HF: mdnt571/nom-char-det)')
print('Dùng local:  cp detector_r34.best.pt train_crop/  rồi')
print('  python train_crop/make_report_pdf.py --ckpt train_crop/detector_r34.best.pt \\')
print('         --manifest evaluation/ver_new/char_detector/detect_manifest.json --out train_crop/ket_qua.pdf')